In [1]:
import os
# Keras backend: "tensorflow" (default) or "torch". Must be set before keras is imported.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

'tensorflow'

In [2]:
import numpy as np
import keras
from keras import ops
import torch

# Layer Normalization

Layer normalization (Ba, Kiros, Hinton, 2016) normalizes **each example on its own**, across its feature dimension.
For one feature vector $x \in \mathbb{R}^H$ (e.g. the hidden state of an RNN at one time step):

$$\mu = \frac{1}{H} \sum_{i=1}^{H} x_i \qquad\qquad \sigma^2 = \frac{1}{H} \sum_{i=1}^{H} (x_i - \mu)^2$$

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \qquad\qquad y_i = \gamma_i \, \hat{x}_i + \beta_i$$

- $\gamma, \beta \in \mathbb{R}^H$ are learned scale and shift (initialized to $1$ and $0$), so the layer can undo the normalization if that helps.
- $\epsilon$ is a small constant for numerical stability.

For a sequence batch of shape $(B, T, H)$, the statistics are computed separately for every $(b, t)$ position — over $H$ only.

### Layer Norm vs Batch Norm

| | Batch Norm | Layer Norm |
|---|---|---|
| statistics over | batch (and time) axis, per feature | feature axis, per example (and time step) |
| depends on other examples in the batch | yes | no |
| running mean/variance for inference | yes | no — same computation in training and inference |
| works with batch size 1 / variable-length sequences | poorly | yes |

That is why RNNs and Transformers use layer norm: the statistics of a time step don't depend on the batch or on other time steps.

## Keras: `keras.layers.LayerNormalization`

A toy batch of $B=2$ sequences, $T=3$ time steps, $H=4$ features.

In [3]:
rng = np.random.default_rng(0)
x = rng.normal(loc=5.0, scale=3.0, size=(2, 3, 4)).astype("float32")
x

array([[[ 5.3771906,  4.6036854,  6.921268 ,  5.3147   ],
        [ 3.3929918,  6.084785 ,  8.912    ,  7.841243 ],
        [ 2.8887942,  1.2037356,  3.1301765,  5.123978 ]],

       [[-1.9750923,  4.343625 ,  1.2622671,  2.8031979],
        [ 3.367223 ,  4.0510993,  6.2348914,  8.12754  ],
        [ 4.614396 ,  9.09939  ,  3.004416 ,  6.05453  ]]], dtype=float32)

In [4]:
# axis=-1 normalizes over the last (feature) axis; epsilon defaults to 1e-3 in Keras
ln = keras.layers.LayerNormalization(axis=-1, epsilon=1e-5)
y = ln(x)
y

<tf.Tensor: shape=(2, 3, 4), dtype=float32, numpy=
array([[[-0.20930862, -1.1239014 ,  1.6164088 , -0.2831974 ],
        [-1.5161127 , -0.2265811 ,  1.1278255 ,  0.6148684 ],
        [-0.14227676, -1.3538632 ,  0.03128123,  1.4648592 ]],

       [[-1.532574  ,  1.1697155 , -0.14807117,  0.5109298 ],
        [-1.1075554 , -0.7430494 ,  0.42091155,  1.4296925 ],
        [-0.48093438,  1.518522  , -1.1986803 ,  0.16109228]]],
      dtype=float32)>

Every $(b, t)$ position now has mean $\approx 0$ and variance $\approx 1$ across its 4 features:

In [5]:
print("mean:\n", np.round(ops.convert_to_numpy(ops.mean(y, axis=-1)), 4))
print("var:\n", np.round(ops.convert_to_numpy(ops.var(y, axis=-1)), 4))

mean:
 [[ 0.  0.  0.]
 [ 0. -0. -0.]]
var:
 [[1. 1. 1.]
 [1. 1. 1.]]


The learned parameters $\gamma$ (`gamma`) and $\beta$ (`beta`) have one entry per feature:

In [6]:
for w in ln.weights:
    print(w.path, w.shape, ops.convert_to_numpy(w))

layer_normalization/gamma (4,) [1. 1. 1. 1.]
layer_normalization/beta (4,) [0. 0. 0. 0.]


### The same computation by hand

In [7]:
eps = 1e-5
mu = x.mean(axis=-1, keepdims=True)
var = x.var(axis=-1, keepdims=True)   # biased variance (divide by H), as in the formula
y_manual = (x - mu) / np.sqrt(var + eps)   # gamma = 1, beta = 0 at initialization

np.allclose(ops.convert_to_numpy(y), y_manual, atol=1e-5)

True

### Independent of the batch

Normalizing the first sequence alone gives the same result as normalizing it inside the batch (not true for batch norm in training mode):

In [8]:
y_single = ln(x[:1])
np.allclose(ops.convert_to_numpy(y_single), ops.convert_to_numpy(y)[:1], atol=1e-6)

True

## PyTorch: `torch.nn.LayerNorm`

`normalized_shape` lists the trailing dimensions to normalize over (here just the feature dimension $H=4$).
The default `eps` is `1e-5` (Keras uses `1e-3`), and $\gamma, \beta$ are called `weight` and `bias`.

In [9]:
ln_torch = torch.nn.LayerNorm(normalized_shape=4, eps=1e-5)
y_torch = ln_torch(torch.from_numpy(x))
y_torch

tensor([[[-0.2093, -1.1239,  1.6164, -0.2832],
         [-1.5161, -0.2266,  1.1278,  0.6149],
         [-0.1423, -1.3539,  0.0313,  1.4649]],

        [[-1.5326,  1.1697, -0.1481,  0.5109],
         [-1.1076, -0.7430,  0.4209,  1.4297],
         [-0.4809,  1.5185, -1.1987,  0.1611]]],
       grad_fn=<NativeLayerNormBackward0>)

In [10]:
for name, p in ln_torch.named_parameters():
    print(name, tuple(p.shape), p.detach().numpy())

weight (4,) [1. 1. 1. 1.]
bias (4,) [0. 0. 0. 0.]


In [11]:
np.allclose(y_torch.detach().numpy(), y_manual, atol=1e-5)

True

There is also a functional form, where $\gamma$ and $\beta$ are passed explicitly:

In [12]:
gamma = torch.tensor([1.0, 2.0, 0.5, 1.0])
beta = torch.tensor([0.0, 1.0, 0.0, -1.0])
y_func = torch.nn.functional.layer_norm(torch.from_numpy(x), normalized_shape=(4,), weight=gamma, bias=beta, eps=1e-5)

np.allclose(y_func.numpy(), y_manual * gamma.numpy() + beta.numpy(), atol=1e-5)

True

## Using it in a Transformer block

Every Transformer block has two sub-layers — self-attention and a position-wise feed-forward network — each wrapped
in a residual connection and a layer norm. Most current models use the **pre-norm** arrangement (normalize the input of each sub-layer):

$$h = x + \mathrm{Attention}(\mathrm{LN}_1(x)) \qquad\qquad y = h + \mathrm{FFN}(\mathrm{LN}_2(h))$$

The original Transformer used **post-norm**, $y = \mathrm{LN}(x + \mathrm{Sublayer}(x))$; pre-norm is easier to train in deep stacks.
Either way, layer norm acts on each token's $d_\text{model}$-dimensional vector independently.

### Keras

In [13]:
def transformer_block(x, d_model: int, num_heads: int, d_ff: int):
    # self-attention sub-layer (pre-norm)
    h = keras.layers.LayerNormalization(epsilon=1e-5)(x)
    h = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)(h, h)
    x = x + h
    # feed-forward sub-layer (pre-norm)
    h = keras.layers.LayerNormalization(epsilon=1e-5)(x)
    h = keras.layers.Dense(d_ff, activation="gelu")(h)
    h = keras.layers.Dense(d_model)(h)
    return x + h

d_model = 16
inputs = keras.Input(shape=(None, 4))                    # (batch, time, features), any sequence length
h = keras.layers.Dense(d_model)(inputs)                  # project features to d_model
h = transformer_block(h, d_model=d_model, num_heads=2, d_ff=32)
h = keras.layers.LayerNormalization(epsilon=1e-5)(h)    # final norm, as in pre-norm models
outputs = keras.layers.Dense(1)(h[:, -1])               # predict from the last time step
model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None, 4)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 16)  │         80 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 16)  │         32 │ dense[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 16)  │      1,088 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 16)  │          0 │ dense[0][0],      │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 16)  │         32 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 32)  │        544 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None, 16)  │        528 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 16)  │          0 │ add[0][0],        │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 16)  │         32 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 16)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         17 │ get_item[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,353 (9.19 KB)

 Trainable params: 2,353 (9.19 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model(x).shape

TensorShape([2, 1])

### PyTorch

In [15]:
class TransformerBlock(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.attn = torch.nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_ff),
            torch.nn.GELU(),
            torch.nn.Linear(d_ff, d_model),
        )

    def forward(self, x):                    # x: (batch, time, d_model)
        h = self.norm1(x)                    # normalizes every token over d_model
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + h
        return x + self.ffn(self.norm2(x))


class TinyTransformer(torch.nn.Module):
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.proj = torch.nn.Linear(n_features, d_model)
        self.block = TransformerBlock(d_model, num_heads=2, d_ff=32)
        self.norm = torch.nn.LayerNorm(d_model)
        self.head = torch.nn.Linear(d_model, 1)

    def forward(self, x):                    # x: (batch, time, features)
        h = self.block(self.proj(x))
        return self.head(self.norm(h)[:, -1])

torch_model = TinyTransformer(n_features=4, d_model=16)
torch_model(torch.from_numpy(x)).shape

torch.Size([2, 1])

PyTorch also ships this block as `torch.nn.TransformerEncoderLayer`; `norm_first=True` selects pre-norm, the default `False` is post-norm:

In [16]:
layer = torch.nn.TransformerEncoderLayer(d_model=16, nhead=2, dim_feedforward=32, batch_first=True, norm_first=True)
print(layer.norm1, layer.norm2)
layer(torch.randn(2, 3, 16)).shape

LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True) LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)


torch.Size([2, 3, 16])

## Variant: RMS Normalization

Many recent Transformers (e.g. Llama) use RMSNorm, which skips the mean subtraction and the shift $\beta$:

$$\mathrm{RMS}(x) = \sqrt{\frac{1}{H} \sum_{i=1}^{H} x_i^2 + \epsilon} \qquad\qquad y_i = \gamma_i \, \frac{x_i}{\mathrm{RMS}(x)}$$

Both frameworks provide it: `keras.layers.RMSNormalization` and `torch.nn.RMSNorm`.

In [17]:
y_rms_manual = x / np.sqrt((x ** 2).mean(axis=-1, keepdims=True) + eps)

y_rms_keras = keras.layers.RMSNormalization(axis=-1, epsilon=eps)(x)
y_rms_torch = torch.nn.RMSNorm(normalized_shape=4, eps=eps)(torch.from_numpy(x))

np.allclose(ops.convert_to_numpy(y_rms_keras), y_rms_manual, atol=1e-5), np.allclose(y_rms_torch.detach().numpy(), y_rms_manual, atol=1e-5)

(True, True)